# Chapter 11: File I/O

## Opening and Reading Files

In [1]:
sample_text = """Python is a very powerful language."""

In [2]:
f = open("sample.txt", "w")
f.write(sample_text)
f.close()

In [3]:
# f = open("sample.txt", "r")
# content = f.read()
# print(f"content: {content}")
# f.close()

## The `with` Statement

In [4]:
# Read entire content as a single string
with open("sample.txt", "r") as f:
    content = f.read()
print(f"content: {content}")

content: Python is a very powerful language.


In [5]:
# Read all lines into a list
with open("sample.txt", "r") as f:
    lines = f.readlines()
print(f"lines: {lines}")

lines: ['Python is a very powerful language.']


In [6]:
# Read one line at a time
with open("sample.txt", "r") as f:
    for line in f:
        print(line.strip())

Python is a very powerful language.


## Writing Files

In [7]:
with open("output.txt", "w") as f:
    f.write("First line\n")
    f.write("Second line\n")

# with open("output.txt", "r") as f:
#     content = f.read()
# print(f"content: {content}")

In [8]:
with open("output.txt", "a") as f:
    f.write("Third line appended\n")

# with open("output.txt", "r") as f:
#     content = f.read()
# print(f"content: {content}")

### Common file modes

- `"r"` - read (default)
- `"w"` - write (creates or overwrites)
- `"a"` - append
- `"x"` - exclusive creation (fails if the file exists)
- `"rb"`, `"wb"` - binary read/write

In [9]:
lines = ["First line\n", "Second line\n", "Third line\n"]
with open("multiline.txt", "w") as f:
    f.writelines(lines)

with open("multiline.txt", "r") as f:
    content = f.read()
print(f"content: {content}")

content: First line
Second line
Third line



## Handling File Errors

In [10]:
def read_file_safe(path):
    try:
        with open(path, "r") as f:
            return f.read()
    except FileNotFoundError:
        print(f"File not found: {path}")
        return None
    except PermissionError:
        print(f"Permission denied: {path}")
        return None
    except OSError as e:
        print(f"OS error reading {path}: {e}")
        return None

In [11]:
read_file_safe("sample.txt")

'Python is a very powerful language.'

In [12]:
read_file_safe("doesnt_exist.txt")

File not found: doesnt_exist.txt


## The Context Manager Protocol

In [13]:
import time


class Timer:
    """A context manager that measures elapsed time."""

    def __init__(self, label=""):
        self.label = label
        self.start = None

    def __enter__(self):
        self.start = time.perf_counter()
        print(f"[{self.label}] Starting timer...")
        return self

    def __exit__(self, exc_type, exc, tb):
        assert self.start is not None
        elapsed = time.perf_counter() - self.start
        print(f"[{self.label}] Elapsed: {elapsed:.6f} s")
        return False

In [14]:
with Timer("computation") as t:
    total = sum(range(1_000_000))
    print(f"Sum: {total}")

[computation] Starting timer...
Sum: 499999500000
[computation] Elapsed: 0.009243 s


## `contextlib.contextmanager`

In [15]:
from contextlib import contextmanager

In [16]:
@contextmanager
def timer(label=""):
    start = time.perf_counter()
    print(f"[{label}] Starting timer...")
    try:
        yield
    finally:
        elapsed = time.perf_counter() - start
        print(f"[{label}] Elapsed: {elapsed:.6f} s")

In [17]:
with timer("computation"):
    total = sum(range(1_000_000))
    print(f"Sum: {total}")

[computation] Starting timer...
Sum: 499999500000
[computation] Elapsed: 0.008273 s


In [18]:
with timer("slow computation"):
    time.sleep(0.1)

[slow computation] Starting timer...
[slow computation] Elapsed: 0.100102 s


In [19]:
@contextmanager
def managed_resource(name):
    print(f"Acquiring {name}")
    try:
        yield name
    except Exception as e:
        print(f"Error while using {name}: {e}")
        raise
    finally:
        print(f"Releasing {name}")

In [20]:
with managed_resource("database connection") as conn:
    print(f"Using {conn}")

Acquiring database connection
Using database connection
Releasing database connection


## Working with File Paths using `pathlib`


In [21]:
from pathlib import Path

In [22]:
p = Path("sample.txt")
data_dir = Path("data")
config = data_dir / "config" / "settings.json"

In [23]:
print(f"Path: {config}")
print(f"Name: {config.name}")
print(f"Stem: {config.stem}")
print(f"Suffix: {config.suffix}")
print(f"Parent: {config.parent}")

Path: data\config\settings.json
Name: settings.json
Stem: settings
Suffix: .json
Parent: data\config


In [24]:
print(f"Exists: {p.exists()}")
print(f"Is file: {p.is_file()}")
# Typo in recording: label should be "Is dir", not "Is file"
print(f"Is dir: {p.is_dir()}")
print(f"Absolute: {p.resolve()}")

Exists: True
Is file: True
Is dir: False
Absolute: C:\Users\Frisu\OneDrive\Desktop\GitHub\IntroductionToPython\ch11_file_io\sample.txt


In [25]:
text = p.read_text()
print(f"text: {text}")

text: Python is a very powerful language.


In [26]:
Path("output_pathlib.txt").write_text("Written via pathlib!")

20